In [ ]:
# --- SCD1 with pyspark ---
class scd1:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, when
    spark = SparkSession.builder.getOrCreate()
    # Existing dimension table
    dim_customer = spark.createDataFrame([
        (1, "Rahul", "Kolkata"),
        (2, "Anita", "Delhi")
    ], ["cust_id", "cust_name", "city"])
    # Incoming updates
    updates = spark.createDataFrame([
        (1, "Rahul", "Rajarhat"),   # City changed
        (2, "Anita", "Delhi")       # No change
    ], ["cust_id", "cust_name", "city"])
    # --- SCD1 with pyspark ---
    # SCD1 logic: overwrite old values with new ones
    updated_df = (
        dim_customer.alias("dim")
        .join(updates.alias("upd"), "cust_id", "left")
        .select(
            col("cust_id"),
            when(col("upd.cust_name").isNotNull(),
                col("upd.cust_name"))
            .otherwise(col("dim.cust_name"))
            .alias("cust_name"),

            when(col("upd.city").isNotNull(),
                col("upd.city"))
            .otherwise(col("dim.city"))
            .alias("city")
        )
    )


    scd1_result.show()

    scd1_result.show()



# SCD using pyspark
# Steps -
#   1. filter only that records from target table with active falg yes in active_flag_df.
#   2. join active_flag_df with source_df in join_df
#   3. filter out and select rows with changed data (filter by comparing tgt.col != src.col) in changed_df
#   4. now closed the records by adding col active_flag as 'No' and end_date as current_date from
#       changed_df.tgt.cols (close only target tbl cols from from changed_df) in closed_df
#   5. select rows from changed_df for src.cols (select source data only, from changed_df) and add active flag
#       as 'Yes', end_date as null in new_df and start date as current_date
#   6. union all both closed_df and new_df to target_df

In [ ]:
class SchemaEvaluationDeltaTableOp7:
    # implimenting schema evaluation

    from pyspark.sql import SparkSession
    from pyspark.sql.functions import expr
    from delta.tables import DeltaTable

    # -------------------------------
    # 1. Spark Session with Delta Support
    # -------------------------------
    spark = SparkSession.builder \
        .appName("SchemaEvolutionETL") \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()

    # -------------------------------
    # 2. Read Incoming Data (New Schema)
    # -------------------------------
    incoming_path = "/data/incoming/orders"
    incoming_df = spark.read.format("parquet").load(incoming_path)

    # -------------------------------
    # 3. Target Delta Table Path
    # -------------------------------
    delta_path = "/delta/orders"

    #---------------
    # 4. Handle Schema Evolution
    # -------------------------------
    # If Delta table exists, merge schema and upsert
    if DeltaTable.isDeltaTable(spark, delta_path):
        delta_table = DeltaTable.forPath(spark, delta_path)

        # Merge logic based on unique key (order_id)
        delta_table.alias("target").merge(
            source=incoming_df.alias("source"),
            condition="target.order_id = source.order_id"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    else:
        # First time write with schema evolution enabled
        incoming_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save(delta_path)

    # -------------------------------
    # 5. Validate Schema Evolution
    # -------------------------------
    print("Current Schema:")
    spark.read.format("delta").load(delta_path).printSchema()


In [ ]:
class scd2_newAppproch:
    # --- SCD2 with Delta Lake MERGE ---
    from pyspark.sql import functions as F, Window
    from delta.tables import DeltaTable

    # ----------------------------
    # CONFIG
    # ----------------------------
    target_path    = "dbfs:/mnt/dim/dim_customer"   # or use a table: "catalog.schema.dim_customer"
    natural_keys   = ["customer_id"]
    tracked_cols   = ["name", "email", "city"]      # change-tracked attributes
    metadata_cols  = ["phone", "state"]             # pass-through attributes (won't trigger SCD2 if only these change)
    soft_delete    = True                           # expire rows not present in source snapshot
    batch_ts       = F.current_timestamp()          # could be your pipeline run timestamp
    start_col      = "effective_start_at"
    end_col        = "effective_end_at"
    flag_col       = "is_current"
    version_col    = "version"
    hash_col       = "change_hash"

    # Your source DataFrame. Replace this with your actual source.
    # df_src = ... (customer_id, name, email, city, phone, state, ...)
    # Ensure one row per natural key at this stage (dedup appropriately).

    # 1) Prepare source with change hash
    def null_safe_str(colname):
        return F.coalesce(F.col(colname).cast("string"), F.lit(""))

    df_src_prepared = (
        df_src
        .withColumn(
            hash_col,
            F.sha2(F.concat_ws("||", *[null_safe_str(c) for c in tracked_cols]), 256)
        )
        .withColumn("batch_ts", batch_ts)
    )

    # 2) Ensure target exists as Delta table with correct schema
    def bootstrap_empty_target():
        # Create an empty DF with expected schema if not exists
        base_cols = natural_keys + tracked_cols + metadata_cols
        df_empty = df_src_prepared.select(*base_cols).limit(0) \
            .withColumn(start_col, F.lit(None).cast("timestamp")) \
            .withColumn(end_col, F.lit(None).cast("timestamp")) \
            .withColumn(flag_col, F.lit(None).cast("boolean")) \
            .withColumn(version_col, F.lit(None).cast("int")) \
            .withColumn(hash_col, F.lit(None).cast("string"))
        df_empty.write.format("delta").mode("overwrite").save(target_path)

    # Initialize Delta target if needed
    if not DeltaTable.isDeltaTable(spark, target_path):
        bootstrap_empty_target()

    delta_tgt = DeltaTable.forPath(spark, target_path)
    df_tgt = delta_tgt.toDF()

    # 3) Join source to current target to figure out new vs changed vs same
    current_tgt = (
        df_tgt
        .filter(F.col(flag_col) == True)
        .select(*natural_keys, hash_col, version_col)
        .withColumnRenamed(hash_col, "tgt_hash")
        .withColumnRenamed(version_col, "tgt_version")
    )

    src_vs_tgt = (
        df_src_prepared.alias("s")
        .join(current_tgt.alias("t"), on=natural_keys, how="left")
        .withColumn("is_new_key", F.col("t.tgt_hash").isNull())
        .withColumn("is_changed", (~F.col("is_new_key")) & (F.col(hash_col) != F.col("t.tgt_hash")))
    )

    # 4) Build "staged" DataFrame for MERGE using the mergeKey trick:
    #    - Rows that EXPIRE current versions (matched updates)
    #    - Rows that INSERT new versions (not matched insert)
    # We'll concatenate natural keys into a single merge string.
    def concat_nk(prefix):
        # prefix "t." or "s."
        return F.concat_ws("||", *[F.col(f"{prefix}{c}").cast("string") for c in natural_keys])

    # a) Rows to expire (matched update)
    expire_df = (
        src_vs_tgt
        .filter("is_changed")
        .select(*[F.col(f"s.{c}").alias(c) for c in df_src_prepared.columns])  # bring s.*
        .withColumn("action", F.lit("expire"))
        .withColumn("mergeKey", concat_nk("s."))  # will match target
        .select("action", "mergeKey", *df_src_prepared.columns)
    )

    # b) Rows to insert (new keys + changed keys)
    insert_df = (
        src_vs_tgt
        .filter("is_new_key OR is_changed")
        .withColumn(version_col, F.when(F.col("is_new_key"), F.lit(1)).otherwise(F.col("t.tgt_version") + F.lit(1)))
        .withColumn(start_col, F.col("batch_ts"))
        .withColumn(end_col, F.lit(None).cast("timestamp"))
        .withColumn(flag_col, F.lit(True))
        .withColumn("action", F.lit("insert"))
        .withColumn("mergeKey", F.lit(None).cast("string"))  # force NOT MATCHED branch
    )

    # Columns to carry into the target on insert
    base_cols = natural_keys + tracked_cols + metadata_cols
    insert_cols = base_cols + [start_col, end_col, flag_col, version_col, hash_col] + ["batch_ts", "action", "mergeKey"]

    staged_updates = (
        expire_df.select("action", "mergeKey", *df_src_prepared.columns)
        .unionByName(insert_df.select(*insert_cols))
    )

    # 5) Perform MERGE: expire matched, insert new versions
    # Build ON condition: concat of target NK equals staged mergeKey (only matches for expire rows)
    on_cond = F.concat_ws("||", *[F.col(f"t.{c}").cast("string") for c in natural_keys]) == F.col("s.mergeKey")

    (
        delta_tgt.alias("t")
        .merge(staged_updates.alias("s"), on_cond)
        # expire old rows
        .whenMatchedUpdate(
            condition="s.action = 'expire' AND t.is_current = true",
            set={
                flag_col: "false",
                end_col: "s.batch_ts"
            }
        )
        # insert new versions (for both new keys and changed rows)
        .whenNotMatchedInsert(
            condition="s.action = 'insert'",
            values={
                **{c: f"s.{c}" for c in base_cols},
                start_col: f"s.{start_col}",
                end_col: f"s.{end_col}",
                flag_col: f"s.{flag_col}",
                version_col: f"s.{version_col}",
                hash_col: f"s.{hash_col}"
            }
        )
        .execute()
    )

    # 6) Optional soft-delete: expire current rows whose business keys vanished from source
    if soft_delete:
        src_keys = df_src_prepared.select(*natural_keys).dropDuplicates()
        current_only = (
            df_tgt.filter(F.col(flag_col) == True)
                .select(*natural_keys)
                .join(src_keys, on=natural_keys, how="left_anti")
                .withColumn("batch_ts", batch_ts)
                .withColumn("mergeKey", concat_nk(""))
                .select("mergeKey", "batch_ts")
        )

        # We only need the key + batch_ts to expire matches
        (
            delta_tgt.alias("t")
            .merge(current_only.alias("s"),
                F.concat_ws("||", *[F.col(f"t.{c}").cast("string") for c in natural_keys]) == F.col("s.mergeKey"))
            .whenMatchedUpdate(
                condition=f"t.{flag_col} = true",
                set={flag_col: "false", end_col: "s.batch_ts"}
            )
            .execute()
        )


In [ ]:
class scd2_newApproch2:    
    from pyspark.sql import functions as F

    # CONFIG (same as above)
    natural_keys   = ["customer_id"]
    tracked_cols   = ["name", "email", "city"]
    metadata_cols  = ["phone", "state"]
    batch_ts       = F.current_timestamp()
    start_col      = "effective_start_at"
    end_col        = "effective_end_at"
    flag_col       = "is_current"
    version_col    = "version"
    hash_col       = "change_hash"

    # df_src (as before)
    df_src_prepared = (
        df_src
        .withColumn(
            hash_col,
            F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in tracked_cols]), 256)
        )
        .withColumn("batch_ts", batch_ts)
    )

    # Read target (Parquet/Hive), if not exists create empty
    # df_tgt = spark.read.parquet(target_path) ... else:
    base_cols = natural_keys + tracked_cols + metadata_cols
    df_empty = df_src_prepared.select(*base_cols).limit(0) \
        .withColumn(start_col, F.lit(None).cast("timestamp")) \
        .withColumn(end_col, F.lit(None).cast("timestamp")) \
        .withColumn(flag_col, F.lit(None).cast("boolean")) \
        .withColumn(version_col, F.lit(None).cast("int")) \
        .withColumn(hash_col, F.lit(None).cast("string"))

    try:
        df_tgt = spark.read.parquet(target_path)
    except:
        df_empty.write.mode("overwrite").parquet(target_path)
        df_tgt = spark.read.parquet(target_path)

    # Current & historical split
    curr_tgt  = df_tgt.filter(F.col(flag_col) == True)
    hist_tgt  = df_tgt.filter(F.col(flag_col) == False)

    # Join to detect changes
    joined = (
        df_src_prepared.alias("s")
        .join(curr_tgt.select(*natural_keys, hash_col, version_col).alias("t"), on=natural_keys, how="left")
        .withColumn("is_new_key", F.col("t."+hash_col).isNull())
        .withColumn("is_changed", (~F.col("is_new_key")) & (F.col("s."+hash_col) != F.col("t."+hash_col)))
    )

    # Expire old versions (current -> historical)
    to_expire = (
        curr_tgt.alias("tgt")
        .join(joined.filter("is_changed").select(*natural_keys).alias("chg"), on=natural_keys, how="inner")
        .withColumn(flag_col, F.lit(False))
        .withColumn(end_col, batch_ts)
    )

    # Insert new versions (new keys + changed)
    to_insert = (
        joined.filter("is_new_key OR is_changed")
        .withColumn(version_col, F.when(F.col("is_new_key"), F.lit(1)).otherwise(F.col("t."+version_col) + 1))
        .select(*[F.col("s."+c).alias(c) for c in base_cols] + ["s."+hash_col]) \
        .withColumn(start_col, batch_ts) \
        .withColumn(end_col, F.lit(None).cast("timestamp")) \
        .withColumn(flag_col, F.lit(True))
    )

    # Keep unchanged current rows as-is
    unchanged_current = (
        curr_tgt.alias("tgt")
        .join(joined.filter(~F.col("is_new_key") & ~F.col("is_changed")).select(*natural_keys).alias("same"),
            on=natural_keys, how="inner")
    )

    # New target = historical + expired_prior + unchanged_current + inserts
    new_target = (
        hist_tgt
        .unionByName(to_expire.select(df_tgt.columns))
        .unionByName(unchanged_current.select(df_tgt.columns))
        .unionByName(to_insert.select(df_tgt.columns))
    )

    # Overwrite target

In [ ]:
class scd2_pyspark:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import StructType, StructField, StringType
    from pyspark.sql.functions import col, lit, when

    # Spark session
    spark = SparkSession.builder.appName("SCD2Example").getOrCreate()

    # Source schema
    source_schema = StructType([
        StructField("Member_Key", StringType(), True),
        StructField("Member_ID", StringType(), True),
        StructField("Member_Name", StringType(), True),
        StructField("City", StringType(), True),
        StructField("Start_Date", StringType(), True)
    ])

    # Source data
    source_data = [
        ("1", "M001", "John Smith",   "New York",  "2020-01-01"),
        ("3", "M002", "Alice Brown",  "Chicago",   "2019-03-15"),
        ("5", "M003", "David Lee",    "San Diego", "2021-05-10"),
        ("6", "M004", "Emma Wilson",  "Miami",     "2020-09-01"),
        ("8", "M005", "Michael Chen", "Dallas",    "2022-07-20"),
        ("9", "M006", "Sarah Davis",  "Denver",    "2019-11-11")
    ]

    df_source = spark.createDataFrame(source_data, source_schema)

    # Target schema
    target_schema = StructType([
        StructField("Member_Key", StringType(), True),
        StructField("Member_ID", StringType(), True),
        StructField("Member_Name", StringType(), True),
        StructField("City", StringType(), True),
        StructField("Start_Date", StringType(), True),
        StructField("End_Date", StringType(), True),
        StructField("Current_Flag", StringType(), True)
    ])

    # Target data
    target_data = [
        ("1", "M001", "John Smith",   "New York",  "2020-01-01", "2021-06-30", "N"),
        ("2", "M001", "John Smith",   "Boston",    "2021-07-01", None,         "Y"),
        ("3", "M002", "Alice Brown",  "Chicago",   "2019-03-15", "2022-02-28", "N"),
        ("4", "M002", "Alice Brown",  "Seattle",   "2022-03-01", None,         "Y"),
        ("5", "M003", "David Lee",    "San Diego", "2021-05-10", None,         "Y"),
        ("6", "M004", "Emma Wilson",  "Miami",     "2020-09-01", "2023-01-15", "N"),
        ("7", "M004", "Emma Wilson",  "Orlando",   "2023-01-16", None,         "Y"),
        ("8", "M005", "Michael Chen", "Dallas",    "2022-07-20", None,         "Y"),
        ("9", "M006", "Sarah Davis",  "Denver",    "2019-11-11", "2021-12-31", "N"),
        ("10","M006", "Sarah Davis",  "Austin",    "2022-01-01", None,         "Y")
    ]

    df_target = spark.createDataFrame(target_data, target_schema)

    # Step 1: Get current active records from target
    df_current = df_target.filter(col("Current_Flag") == "Y")

    # Step 2: Join source with current target on Member_ID
    df_join = df_source.alias("src").join(
        df_current.alias("tgt"), col("src.Member_ID") == col("tgt.Member_ID"), "left")

    # Step 3: Detect changes in City
    df_changes = df_join.filter(col("src.City") != col("tgt.City"))    

    # Step 4: Close old record (set End_Date and Current_Flag = 'N')
    df_closed = df_changes.select(
        col("tgt.Member_Key"),
        col("tgt.Member_ID"),
        col("tgt.Member_Name"),
        col("tgt.City"),
        col("tgt.Start_Date"),
        lit("2025-11-25").alias("End_Date"),  # Assume today's date
        lit("N").alias("Current_Flag")
    )

    # Step 5: Insert new record with updated City
    df_new = df_changes.select(
        lit(None).alias("Member_Key"),  # Surrogate key can be generated later
        col("src.Member_ID"),
        col("src.Member_Name"),
        col("src.City"),
        lit("2025-11-25").alias("Start_Date"),  # Assume today's date
        lit(None).alias("End_Date"),
        lit("Y").alias("Current_Flag")
    )

    # Step 6: Union old + new + unchanged records
    df_final = df_target.union(df_closed).union(df_new)
